# Data Exploration
Exploratory data analysis: class distribution, image sizes, bounding box statistics, annotation quality.

**Run this notebook before training** to identify issues early.

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
from collections import Counter

import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import pandas as pd
import yaml

with open('../config/config.yaml') as f:
    CFG = yaml.safe_load(f)

CLASS_NAMES = CFG['classes']
IMAGES_DIR  = Path('../data/raw')
LABELS_DIR  = Path('../data/annotations')

print('Classes:', CLASS_NAMES)
print('Num classes:', len(CLASS_NAMES))

## 1. Dataset Overview

In [ ]:
image_paths = sorted(IMAGES_DIR.glob('**/*'))
image_paths = [p for p in image_paths if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}]

label_paths = sorted(LABELS_DIR.glob('*.txt'))
print(f'Images found : {len(image_paths)}')
print(f'Labels found : {len(label_paths)}')
print(f'Missing labels: {len(image_paths) - len(label_paths)}')

## 2. Class Distribution

In [ ]:
class_counts = Counter()

for lbl_path in label_paths:
    for line in lbl_path.read_text().strip().splitlines():
        parts = line.strip().split()
        if parts:
            class_counts[int(parts[0])] += 1

labels = [CLASS_NAMES[k] if k < len(CLASS_NAMES) else str(k) for k in sorted(class_counts)]
counts = [class_counts[k] for k in sorted(class_counts)]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(labels, counts, color=plt.cm.tab20.colors[:len(labels)])
ax.set_xlabel('Class')
ax.set_ylabel('Annotation Count')
ax.set_title('Class Distribution')
plt.xticks(rotation=35, ha='right')
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(count), ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

df_counts = pd.DataFrame({'class': labels, 'count': counts})
df_counts['pct'] = (df_counts['count'] / df_counts['count'].sum() * 100).round(1)
print(df_counts.to_string(index=False))

## 3. Image Size Distribution

In [ ]:
widths, heights = [], []
for img_path in image_paths[:200]:  # sample first 200 for speed
    img = cv2.imread(str(img_path))
    if img is not None:
        h, w = img.shape[:2]
        widths.append(w)
        heights.append(h)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.hist(widths, bins=20, color='steelblue', edgecolor='white')
ax1.set_title('Image Width Distribution')
ax1.set_xlabel('Width (px)')

ax2.hist(heights, bins=20, color='coral', edgecolor='white')
ax2.set_title('Image Height Distribution')
ax2.set_xlabel('Height (px)')

plt.tight_layout()
plt.show()
print(f'Width  — min: {min(widths)}, max: {max(widths)}, median: {int(np.median(widths))}')
print(f'Height — min: {min(heights)}, max: {max(heights)}, median: {int(np.median(heights))}')

## 4. Bounding Box Size Distribution

In [ ]:
bbox_areas = []
bbox_wh_ratios = []

for lbl_path in label_paths:
    for line in lbl_path.read_text().strip().splitlines():
        parts = line.strip().split()
        if len(parts) >= 5:
            _, _, _, w, h = parts[:5]
            w, h = float(w), float(h)
            bbox_areas.append(w * h)
            bbox_wh_ratios.append(w / h if h > 0 else 0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.hist(bbox_areas, bins=30, color='mediumseagreen', edgecolor='white')
ax1.set_title('Normalised BBox Area Distribution')
ax1.set_xlabel('w × h (normalised)')

ax2.hist(bbox_wh_ratios, bins=30, color='mediumpurple', edgecolor='white')
ax2.set_title('BBox Width/Height Ratio')
ax2.set_xlabel('w / h')

plt.tight_layout()
plt.show()
print(f'Median box area: {np.median(bbox_areas):.4f}')
print(f'Very small boxes (<0.005): {sum(1 for a in bbox_areas if a < 0.005)}')

## 5. Annotation Quality Spot-Check

In [ ]:
import random

sample_imgs = random.sample(image_paths, min(6, len(image_paths)))
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

COLORS = plt.cm.tab10.colors

for ax, img_path in zip(axes.flat, sample_imgs):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    ax.imshow(img)
    lbl_path = LABELS_DIR / img_path.with_suffix('.txt').name
    if lbl_path.exists():
        for line in lbl_path.read_text().strip().splitlines():
            parts = line.strip().split()
            if len(parts) >= 5:
                cid, cx, cy, bw, bh = int(parts[0]), float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
                x1 = (cx - bw/2) * w
                y1 = (cy - bh/2) * h
                rect = patches.Rectangle((x1, y1), bw*w, bh*h,
                                          linewidth=2, edgecolor=COLORS[cid % len(COLORS)], facecolor='none')
                ax.add_patch(rect)
                cls_name = CLASS_NAMES[cid] if cid < len(CLASS_NAMES) else str(cid)
                ax.text(x1, y1-3, cls_name, color=COLORS[cid % len(COLORS)], fontsize=7)
    ax.set_title(img_path.name[:30], fontsize=8)
    ax.axis('off')

plt.suptitle('Annotation Quality Spot-Check (random sample)', y=1.02)
plt.tight_layout()
plt.show()